# AI-Inferred 3D Video Reconstruction — Google Colab

This notebook runs the same shared `src/`, `blender/`, and `config/` pipeline used by the local application. After reconstruction completes, it embeds the shared judge result UI directly in Colab without a public tunnel or duplicated pipeline logic.

Before running: push the latest project changes to GitHub, select **Runtime → Change runtime type → GPU**, then run the cells in order. Completed Blender gaps and final results are copied to Google Drive. Colab hardware and session lifetime are not guaranteed.

In [ ]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/redmas45/3D_reconstruction.git"
REPOSITORY_BRANCH = "main"
PROJECT_DIRECTORY = Path("/content/3D_reconstruction")
DRIVE_ROOT = Path("/content/drive/MyDrive/3D_Reconstruction")
BLENDER_VERSION = "4.5.10"
BLENDER_ARCHIVE_URL = (
    f"https://download.blender.org/release/Blender4.5/"
    f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
)
RANDOM_SEED = 42
WORKBENCH_PARALLEL_GAP_RENDERERS = 2
CYCLES_GPU_PARALLEL_GAP_RENDERERS = 1
MAX_PARALLEL_GAP_RENDERERS = WORKBENCH_PARALLEL_GAP_RENDERERS
BLENDER_STALL_TIMEOUT_SECONDS = 900
COLAB_RENDER_ENGINE = "BLENDER_WORKBENCH"
COLAB_CYCLES_COMPUTE_DEVICE = "CUDA"
COLAB_CYCLES_SAMPLES = 2
COLAB_CYCLES_USE_DENOISING = True
PREFERRED_CYCLES_COMPUTE_DEVICES = ("OPTIX", "CUDA")
CYCLES_BENCHMARK_TIMEOUT_SECONDS = 180
COLAB_PRODUCTION_SCALE_PERCENT = 40
COLAB_RECONSTRUCTION_FPS = 6
COLAB_MAX_RENDER_ENTITIES = 6
COLAB_MAXIMUM_PREDICTED_RENDER_SECONDS = 7200
RENDERER_MODE = "blender"

print("Notebook configuration loaded.")

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU was assigned. Change the Colab runtime type to GPU and reconnect.")

subprocess.run(["nvidia-smi"], check=True)


In [ ]:
import os

if (PROJECT_DIRECTORY / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", REPOSITORY_BRANCH], cwd=PROJECT_DIRECTORY, check=True)
    subprocess.run(["git", "checkout", REPOSITORY_BRANCH], cwd=PROJECT_DIRECTORY, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH], cwd=PROJECT_DIRECTORY, check=True)
elif PROJECT_DIRECTORY.exists() and any(PROJECT_DIRECTORY.iterdir()):
    raise RuntimeError(f"Refusing to overwrite non-repository directory: {PROJECT_DIRECTORY}")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(PROJECT_DIRECTORY)],
        check=True,
    )

os.chdir(PROJECT_DIRECTORY)
print(f"Project ready at {PROJECT_DIRECTORY}")


In [ ]:
import shlex
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot use the assigned CUDA GPU after dependency installation.")
print(f"YOLO CUDA device: {torch.cuda.get_device_name(0)}")
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True,
)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "ffmpeg", "xvfb", "xauth", "libgl1", "libxi6", "libxrender1"],
    check=True,
)

blender_install_root = Path("/content/blender")
blender_binary = blender_install_root / f"blender-{BLENDER_VERSION}-linux-x64" / "blender"
if not blender_binary.is_file():
    blender_install_root.mkdir(parents=True, exist_ok=True)
    archive_path = blender_install_root / "blender.tar.xz"
    subprocess.run(["curl", "--fail", "--location", BLENDER_ARCHIVE_URL, "--output", str(archive_path)], check=True)
    subprocess.run(["tar", "-xf", str(archive_path), "-C", str(blender_install_root)], check=True)

if not blender_binary.is_file():
    raise FileNotFoundError(f"Blender executable was not installed at {blender_binary}")

blender_wrapper = blender_install_root / "blender-colab"
blender_wrapper.write_text(
    "#!/usr/bin/env bash\nexec xvfb-run -a " + shlex.quote(str(blender_binary)) + " \"$@\"\n",
    encoding="utf-8",
)
blender_wrapper.chmod(0o755)
os.environ["BLENDER_EXECUTABLE"] = str(blender_wrapper)
subprocess.run([str(blender_wrapper), "--background", "--version"], check=True)
device_probe_command = [
    str(blender_wrapper), "--factory-startup", "--python-expr",
    ("import bpy, gpu; "
     "print('BLENDER_GRAPHICS_VENDOR=' + gpu.platform.vendor_get()); "
     "print('BLENDER_GRAPHICS_RENDERER=' + gpu.platform.renderer_get()); "
     "bpy.ops.wm.quit_blender()"),
]
try:
    device_probe = subprocess.run(
        device_probe_command, capture_output=True, text=True, timeout=60, check=False,
    )
    probe_output = device_probe.stdout + device_probe.stderr
except subprocess.TimeoutExpired:
    probe_output = ""
probe_lines = [line for line in probe_output.splitlines() if line.startswith('BLENDER_GRAPHICS_')]
if probe_lines:
    print('\n'.join(probe_lines))
else:
    print("WARNING: Blender graphics probing was unavailable; reconstruction can still continue.")
if not any("NVIDIA" in line.upper() for line in probe_lines):
    print("Blender display rendering uses software OpenGL; probing Cycles CUDA/OptiX separately.")
cycles_benchmark_output = Path("/content/cycles_gpu_benchmark.png")
for compute_device in PREFERRED_CYCLES_COMPUTE_DEVICES:
    cycles_benchmark_command = [
        str(blender_wrapper), "--background", "--python",
        str(PROJECT_DIRECTORY / "blender" / "probe_cycles.py"), "--",
        "--device", compute_device, "--output", str(cycles_benchmark_output),
    ]
    try:
        cycles_benchmark = subprocess.run(
            cycles_benchmark_command, capture_output=True, text=True,
            timeout=CYCLES_BENCHMARK_TIMEOUT_SECONDS, check=False,
        )
    except subprocess.TimeoutExpired:
        print(f"Cycles {compute_device} benchmark timed out; trying the next renderer.")
        continue
    benchmark_lines = [
        line for line in (cycles_benchmark.stdout + cycles_benchmark.stderr).splitlines()
        if line.startswith("CYCLES_GPU_READY")
    ]
    if cycles_benchmark.returncode == 0 and benchmark_lines and cycles_benchmark_output.is_file():
        COLAB_RENDER_ENGINE = "CYCLES"
        COLAB_CYCLES_COMPUTE_DEVICE = compute_device
        MAX_PARALLEL_GAP_RENDERERS = CYCLES_GPU_PARALLEL_GAP_RENDERERS
        print(benchmark_lines[-1])
        break
    print(f"Cycles {compute_device} is unavailable; trying the next renderer.")
else:
    print("WARNING: Cycles GPU benchmark failed; using the Workbench CPU fallback.")
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL)
print("Python dependencies, FFmpeg, and Blender are ready.")


In [ ]:
from google.colab import drive, files

drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

uploaded_files = files._upload_files(multiple=False)
if len(uploaded_files) != 1:
    raise ValueError("Upload exactly one video file.")

uploaded_name, uploaded_bytes = next(iter(uploaded_files.items()))
safe_name = Path(uploaded_name).name
supported_extensions = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm", ".mpeg", ".mpg", ".wmv"}
if Path(safe_name).suffix.lower() not in supported_extensions:
    raise ValueError(f"Unsupported video extension: {Path(safe_name).suffix}")

input_directory = Path("/content/reconstruction_input")
input_directory.mkdir(parents=True, exist_ok=True)
input_video = input_directory / safe_name
input_video.write_bytes(uploaded_bytes)
del uploaded_bytes, uploaded_files
print(f"Input ready: {input_video.name} ({input_video.stat().st_size / (1024 ** 2):.1f} MB)")


In [ ]:
import copy
import hashlib
import html
import json
import random
import threading
from collections import deque

import ipywidgets as widgets
from IPython.display import Video, display
from google.colab import userdata

AZURE_SECRET_NAMES = (
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_BASE_URL",
    "AZURE_OPENAI_CHAT_DEPLOYMENT",
)
MAXIMUM_ENV_BYTES = 64 * 1024

def colab_secret(secret_name: str) -> str:
    try:
        return str(userdata.get(secret_name) or "").strip()
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return ""

def parse_uploaded_env(payload: bytes) -> dict[str, str]:
    if len(payload) > MAXIMUM_ENV_BYTES:
        raise ValueError("The uploaded .env file is unexpectedly large.")
    values: dict[str, str] = {}
    for line_number, raw_line in enumerate(payload.decode("utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[7:].strip()
        name, separator, value = line.partition("=")
        name = name.strip()
        if not separator:
            raise ValueError(f"Invalid .env assignment on line {line_number}.")
        if name not in AZURE_SECRET_NAMES:
            continue
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '\"'}:
            value = value[1:-1]
        values[name] = value
    return values

azure_values = {
    secret_name: os.environ.get(secret_name, "").strip() or colab_secret(secret_name)
    for secret_name in AZURE_SECRET_NAMES
}
missing_names = [name for name, value in azure_values.items() if not value]
if missing_names:
    print("Upload your local .env file. It is parsed in memory and is not copied to Drive or project outputs.")
    uploaded_env_files = files._upload_files(multiple=False)
    if len(uploaded_env_files) != 1:
        raise ValueError("Upload exactly one .env file.")
    env_name, env_payload = next(iter(uploaded_env_files.items()))
    if Path(env_name).name != ".env":
        raise ValueError("The Azure credential upload must be named .env.")
    uploaded_values = parse_uploaded_env(env_payload)
    for secret_name in missing_names:
        azure_values[secret_name] = uploaded_values.get(secret_name, "")
    del env_payload, uploaded_env_files, uploaded_values
missing_names = [name for name, value in azure_values.items() if not value]
if missing_names:
    raise RuntimeError("Missing required Azure values: " + ", ".join(missing_names))
for secret_name, secret_value in azure_values.items():
    os.environ[secret_name] = secret_value
del azure_values
print("Azure reasoning values loaded in memory without displaying or persisting them.")

source_directory = PROJECT_DIRECTORY / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

from application.reconstruction_pipeline import PipelineOptions, load_config, process_video
from domain.configuration import validate_configuration
from domain.render_runtime_budget import (
    RepresentativePreviewApprovalRequired,
    approve_representative_preview,
)
from infrastructure.azure_openai_reasoner import AzureReasoningSettings, probe_azure_reasoning

configuration = copy.deepcopy(load_config(PROJECT_DIRECTORY / "config" / "reconstruction_config.json"))
configuration["renderer"]["max_parallel_gap_renders"] = MAX_PARALLEL_GAP_RENDERERS
configuration["renderer"]["gap_render_stall_timeout_seconds"] = BLENDER_STALL_TIMEOUT_SECONDS
configuration["renderer"]["engine"] = COLAB_RENDER_ENGINE
configuration["renderer"]["cycles_compute_device"] = COLAB_CYCLES_COMPUTE_DEVICE
configuration["renderer"]["cycles_samples"] = COLAB_CYCLES_SAMPLES
configuration["renderer"]["cycles_use_denoising"] = COLAB_CYCLES_USE_DENOISING
configuration["renderer"]["production_scale_percent"] = COLAB_PRODUCTION_SCALE_PERCENT
configuration["renderer"]["target_fps"] = COLAB_RECONSTRUCTION_FPS
configuration["renderer"]["scale_percent"] = COLAB_PRODUCTION_SCALE_PERCENT
configuration["renderer"]["maximum_predicted_render_seconds"] = COLAB_MAXIMUM_PREDICTED_RENDER_SECONDS
configuration["renderer"]["runtime_budget_enabled"] = True
configuration["renderer"]["allow_runtime_budget_override"] = False
configuration["renderer"]["interactive_preview_approval"] = True
configuration["scene"]["max_render_entities"] = COLAB_MAX_RENDER_ENTITIES
validate_configuration(configuration)
azure_settings = AzureReasoningSettings.from_environment(configuration["reasoning"])
if azure_settings is None:
    raise RuntimeError("Azure reasoning values were not loaded.")
azure_probe = probe_azure_reasoning(azure_settings)
print(f"Azure deployment ready: {azure_probe['deployment']} (response {azure_probe['response_id']})")
with input_video.open("rb") as input_stream:
    video_digest = hashlib.file_digest(input_stream, "sha256").hexdigest()[:12]
checkpoint_configuration = copy.deepcopy(configuration)
checkpoint_configuration["renderer"].pop("max_parallel_gap_renders", None)
checkpoint_configuration["renderer"].pop("gap_render_stall_timeout_seconds", None)
configuration_digest = hashlib.sha256(
    json.dumps(checkpoint_configuration, sort_keys=True).encode()
).hexdigest()[:10]
git_commit = subprocess.check_output(["git", "rev-parse", "--short=8", "HEAD"], cwd=PROJECT_DIRECTORY, text=True).strip()
run_identifier = (
    f"{input_video.stem}_{video_digest}_seed{RANDOM_SEED}_"
    f"cfg{configuration_digest}_git{git_commit}_blender{BLENDER_VERSION}"
)
local_output = Path("/content/reconstruction_runs") / run_identifier
drive_run = DRIVE_ROOT / run_identifier
drive_checkpoint_gaps = drive_run / "checkpoints" / "gaps"
work_directory = local_output / "_work" / f"{input_video.stem}_{video_digest}"
local_gaps = work_directory / "gaps"

if drive_checkpoint_gaps.is_dir():
    shutil.copytree(drive_checkpoint_gaps, local_gaps, dirs_exist_ok=True)
    print(f"Restored completed-gap checkpoints from {drive_checkpoint_gaps}")

options = PipelineOptions(configuration, local_output, reuse_work=True, renderer_mode=RENDERER_MODE)

progress_bar = widgets.FloatProgress(value=0, min=0, max=100, description="Progress", bar_style="info")
stage_label = widgets.HTML(value="<b>Waiting to start</b>")
activity_output = widgets.HTML(
    value="<i>No activity yet</i>",
    layout={"border": "1px solid #475569", "padding": "8px", "max_height": "260px", "overflow_y": "auto"},
)
display(widgets.VBox([stage_label, progress_bar, activity_output]))

activity_lock = threading.Lock()
checkpoint_lock = threading.Lock()
activity_lines = deque(maxlen=16)
checkpointed_gaps: set[str] = set()
last_displayed_percentage = -1

def checkpoint_completed_gaps() -> None:
    if not local_gaps.is_dir():
        return
    with checkpoint_lock:
        for gap_directory in sorted(local_gaps.glob("gap_*")):
            blender_directory = gap_directory / "blender"
            required = [blender_directory / "gap_blender.mp4", blender_directory / "render_report.json", blender_directory / "scene.blend"]
            if gap_directory.name in checkpointed_gaps or not all(path.is_file() for path in required):
                continue
            checkpoint_destination = drive_checkpoint_gaps / gap_directory.name
            checkpoint_temporary = drive_checkpoint_gaps / f"{gap_directory.name}.tmp"
            if checkpoint_temporary.exists():
                shutil.rmtree(checkpoint_temporary)
            shutil.copytree(gap_directory, checkpoint_temporary)
            if checkpoint_destination.exists():
                shutil.rmtree(checkpoint_destination)
            checkpoint_temporary.replace(checkpoint_destination)
            checkpointed_gaps.add(gap_directory.name)

def checkpoint_sparse_frames() -> None:
    if not local_gaps.is_dir():
        return
    with checkpoint_lock:
        for frame_directory in local_gaps.glob("gap_*/blender/renders/frames_*"):
            relative = frame_directory.relative_to(local_gaps)
            destination = drive_checkpoint_gaps / relative
            destination.mkdir(parents=True, exist_ok=True)
            for frame_path in frame_directory.glob("frame_*.png"):
                destination_frame = destination / frame_path.name
                if destination_frame.is_file() and destination_frame.stat().st_size == frame_path.stat().st_size:
                    continue
                shutil.copy2(frame_path, destination_frame)
            manifest_path = frame_directory / "frame_manifest.json"
            if manifest_path.is_file():
                shutil.copy2(manifest_path, destination / manifest_path.name)

def watch_for_checkpoints(stop_event: threading.Event) -> None:
    while not stop_event.wait(30):
        checkpoint_sparse_frames()
        checkpoint_completed_gaps()

def report_progress(stage: str, progress: float, detail: str) -> None:
    global last_displayed_percentage
    percentage = round(progress * 100)
    with activity_lock:
        progress_bar.value = percentage
        stage_label.value = f"<b>{stage.replace('_', ' ').title()}</b> &mdash; {html.escape(detail)}"
        if percentage != last_displayed_percentage:
            activity_lines.append(f"[{percentage:3d}%] {detail}")
            last_displayed_percentage = percentage
            activity_output.value = "<br>".join(html.escape(line) for line in activity_lines)

checkpoint_stop_event = threading.Event()
checkpoint_thread = threading.Thread(
    target=watch_for_checkpoints, args=(checkpoint_stop_event,), name="drive-checkpoint", daemon=True
)
checkpoint_thread.start()
render_device_label = (
    COLAB_CYCLES_COMPUTE_DEVICE if COLAB_RENDER_ENGINE == "CYCLES" else "software graphics"
)
print(
    f"Running {RENDERER_MODE} reconstruction with {COLAB_RENDER_ENGINE}, "
    f"adaptive sharp resolution, {render_device_label}, "
    f"and up to {MAX_PARALLEL_GAP_RENDERERS} Blender workers."
)
try:
    while True:
        try:
            final_output = process_video(input_video, options, random.Random(RANDOM_SEED), report_progress)
            break
        except RepresentativePreviewApprovalRequired as approval_request:
            checkpoint_sparse_frames()
            checkpoint_completed_gaps()
            print("\nThe heaviest representative gap is ready. Review it before spending time on all gaps:")
            display(Video(str(approval_request.preview_path), embed=True))
            approval_answer = input("Type APPROVE to continue the full render, or anything else to stop: ").strip()
            if approval_answer != "APPROVE":
                raise RuntimeError("Full render stopped because the representative preview was not approved.")
            approve_representative_preview(
                approval_request.approval_path,
                approval_request.signature,
            )
            print("Preview approved. Resuming from cached analysis and the saved representative gap.")
finally:
    checkpoint_stop_event.set()
    checkpoint_thread.join(timeout=15)
    checkpoint_sparse_frames()
    checkpoint_completed_gaps()
print(f"Reconstruction finished: {final_output}")


In [ ]:
from IPython.display import Video

def print_reasoning_report(reasoning_path: Path) -> None:
    if not reasoning_path.is_file():
        print("\nEvidence/story report is unavailable; inspect the copied JSON reports for diagnostics.")
        return
    try:
        reasoning = json.loads(reasoning_path.read_text(encoding="utf-8"))
    except (OSError, UnicodeError, json.JSONDecodeError) as error:
        print(f"\nCould not read evidence/story report: {type(error).__name__}")
        return

    print("\n" + "=" * 76)
    print("EVIDENCE-GROUNDED RECONSTRUCTION REPORT")
    print("=" * 76)
    mode = reasoning.get("mode", "unknown")
    deployment = reasoning.get("deployment") or "deterministic fallback"
    print(f"Planner: {mode} ({deployment})")
    if reasoning.get("warning"):
        print(f"Warning: {reasoning['warning']}")
    if reasoning.get("headline"):
        print(f"\n{reasoning['headline']}")
    if reasoning.get("whole_video_summary"):
        confidence = round(float(reasoning.get("confidence", 0.0)) * 100)
        print(f"Summary ({confidence}% support): {reasoning['whole_video_summary']}")
    causal_label = (
        "Evidence-supported" if reasoning.get("causal_link_supported")
        else "Not claimed; sequence is inferred"
    )
    print(f"Causal interpretation: {causal_label}")

    print("\nVISIBLE-EVIDENCE CLUES")
    clues = reasoning.get("clues") or [
        {"statement": statement, "confidence": None}
        for statement in reasoning.get("scene_clues", [])
    ]
    for clue_index, clue in enumerate(clues, start=1):
        confidence = clue.get("confidence")
        confidence_text = f" [{round(float(confidence) * 100)}%]" if confidence is not None else ""
        print(f"  {clue_index:02d}. {clue.get('statement', 'Unnamed clue')}{confidence_text}")

    gap_decisions = {
        int(item["gap_index"]): item for item in reasoning.get("decisions", [])
        if isinstance(item, dict) and isinstance(item.get("gap_index"), int)
    }
    print("\nGAP-BY-GAP STORY")
    for gap in reasoning.get("gap_summaries", []):
        gap_index = int(gap["gap_index"])
        print(f"\n  Gap {gap_index + 1} ({round(float(gap.get('confidence', 0.0)) * 100)}% confidence)")
        print(f"    Before observed: {gap.get('before_observed', 'Unknown')}")
        print(f"    Inside inferred: {gap.get('inside_inferred', 'Unknown')}")
        print(f"    After observed:  {gap.get('after_observed', 'Unknown')}")
        decision = gap_decisions.get(gap_index, {})
        for entity in decision.get("entities", []):
            hypothesis = str(entity.get("selected_hypothesis_id", "unknown")).replace("_", " ")
            entity_confidence = round(float(entity.get("confidence", 0.0)) * 100)
            print(f"      - {entity.get('entity_id', 'entity')}: {hypothesis} ({entity_confidence}%)")
        for unknown in gap.get("unknowns", []):
            print(f"      ? {unknown}")

    print("\nThis report describes AI-inferred evidence visualization, not recovered ground truth.")
    print("=" * 76)

print_reasoning_report(work_directory / "reasoning_public.json")

result_directory = drive_run / "result"
result_directory.mkdir(parents=True, exist_ok=True)
drive_video = result_directory / final_output.name
shutil.copy2(final_output, drive_video)

report_directory = result_directory / "reports"
report_directory.mkdir(parents=True, exist_ok=True)
for report_path in work_directory.rglob("*.json"):
    relative_path = report_path.relative_to(work_directory)
    destination = report_directory / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(report_path, destination)

print(f"Saved final video to Google Drive: {drive_video}")
print(f"Saved JSON evidence/story reports to: {report_directory}")

presentation_manifest = work_directory / "presentation_manifest.json"
if not presentation_manifest.is_file():
    raise RuntimeError("The judge presentation manifest was not created.")
judge_preview = local_output / "judge_preview.mp4"
subprocess.run([
    "ffmpeg", "-y", "-i", str(final_output),
    "-vf", "scale='min(960,iw)':-2:flags=lanczos,fps=24",
    "-c:v", "libx264", "-preset", "veryfast", "-crf", "27",
    "-an", "-movflags", "+faststart", str(judge_preview),
], check=True)

from google.colab import output
from interfaces.http.result_viewer import start_result_viewer

result_server, result_server_thread = start_result_viewer(
    judge_preview, presentation_manifest, PROJECT_DIRECTORY / "web",
)
print("Judge result viewer ready below. Gap markers jump to each reconstructed interval.")
output.serve_kernel_port_as_iframe(result_server.server_port, height=1100)
print("The Colab proxy is authenticated by your notebook session; no public tunnel is opened.")
files.download(str(final_output))


## Troubleshooting

- If the GPU check fails, reconnect using a GPU runtime.
- If cloning does not include recent changes, push the local branch to GitHub and rerun the clone cell.
- If Colab disconnects, reconnect and run the cells again with the same video and seed. Completed Blender gap checkpoints will be restored from Drive.
- Blender display graphics and Cycles compute are checked separately. A successful OptiX/CUDA render probe selects Cycles on the T4; otherwise the notebook falls back to Workbench software rendering.
- Cycles uses one Blender worker because all gaps share one T4. The Workbench fallback uses two CPU workers.
- Do not raise the Cycles worker count above one on a single T4. Parallel GPU processes compete for the same VRAM and commonly reduce reliability.
- Static-camera footage is the supported profile. Moving-camera footage is rendered experimentally and is marked as such in reports.
- The output is an AI-inferred evidence visualization, not recovered ground truth.